# 모델링 공통 설정 (Config)

**이 노트북을 먼저 실행하고, 아래 셀들을 각자의 모델링 노트북 맨 위에 그대로 복붙해서 쓰세요.**

4명이 각자 모델을 만들되, 아래 값들(데이터 파일, random_state, 타겟 이름, 피처 목록, 평가 함수, 스케일링 방식, 탐색할 하이퍼파라미터 범위)은
**절대 각자 임의로 바꾸지 않고 그대로 재사용**하는 것이 원칙입니다.
그래야 "모델이 달라서 성능이 다른 것"과 "설정이 달라서 성능이 다른 것"을 구분할 수 있습니다.

피처를 추가/제거하고 싶으면 개인 판단으로 하지 말고 팀 전체에 먼저 공유하세요.

In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)

## 1. 랜덤 시드 고정

In [15]:
# RandomForest, LightGBM, KFold 등 랜덤성이 개입하는 모든 곳에 이 값을 그대로 쓰세요.
RANDOM_STATE = 42

## 2. 데이터 파일 (고정)

각자 `train_test_split`을 다시 돌리지 마세요. 이미 나눠둔 아래 파일을 그대로 불러다 씁니다.


In [18]:
DATA_DIR = "./crawl_data/"  # 파일 위치에 맞게 수정

X_train = pd.read_csv(DATA_DIR + "X_train_clean.csv")
X_test = pd.read_csv(DATA_DIR + "X_test_clean.csv")
y_train = pd.read_csv(DATA_DIR + "y_train.csv")
y_test = pd.read_csv(DATA_DIR + "y_test.csv")

# activity_id는 식별자일 뿐 피처가 아니므로 모델 입력에서 제외
ID_COL = "activity_id"
if ID_COL in X_train.columns:
    X_train = X_train.drop(columns=[ID_COL])
if ID_COL in X_test.columns:
    X_test = X_test.drop(columns=[ID_COL])

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)

X_train: (5811, 78) | X_test: (1453, 78)
y_train: (5811, 2) | y_test: (1453, 2)


## 3. 타겟 이름 (고정)

철자·괄호·띄어쓰기까지 정확히 이 이름 그대로 씁니다.


In [19]:
TARGET_A = "daily_views_log1p"     # 일평균 조회수(로그변환) - 노출 예측
TARGET_B = "scrap_rate (%)"        # 스크랩 전환율 - 관심도 예측

y_train_A = y_train[TARGET_A]
y_train_B = y_train[TARGET_B]
y_test_A = y_test[TARGET_A]
y_test_B = y_test[TARGET_B]

print("TARGET_A 샘플:", y_train_A.head(3).tolist())
print("TARGET_B 샘플:", y_train_B.head(3).tolist())

TARGET_A 샘플: [2.6799, 3.3646, 3.2504]
TARGET_B 샘플: [0.0, 3.5813, 2.957]


## 4. 피처 목록 (고정, 78개)

`X_train_clean.csv`의 컬럼을 그대로 씁니다. 이 목록과 다른 피처셋으로 학습한 모델은
다른 모델과 성능 비교가 불가능하니, 목록이 이것과 같은지 항상 확인하세요.


In [20]:
FEATURE_COLUMNS = X_train.columns.tolist()

print("피처 개수:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)

피처 개수: 78
['activity_period_missing', 'recruit_period_days', 'recruit_end_dow', 'recruit_end_month', 'activity_period_months', 'title_length', 'title_has_bracket', 'title_has_exclaim', 'title_has_number', 'preferred_count', 'benefit_count', 'extra_benefit_exists', 'activity_period_indefinite', 'company_type_금융권', 'company_type_기타', 'company_type_대기업', 'company_type_동아리/학생자치단체', 'company_type_병원', 'company_type_비영리단체/협회/재단', 'company_type_스타트업', 'company_type_외국계기업', 'company_type_중견기업', 'company_type_중소기업', 'target_대학생', 'target_대학생, 직장인/일반인', 'target_직장인/일반인', 'target_청소년', 'target_청소년, 대학생', 'target_청소년, 직장인/일반인', 'activity_region_경기', 'activity_region_경상', 'activity_region_광주', 'activity_region_대구', 'activity_region_대전', 'activity_region_부산', 'activity_region_서울', 'activity_region_세종', 'activity_region_수도권', 'activity_region_영남권', 'activity_region_울산', 'activity_region_인천', 'activity_region_전국/제한없음', 'activity_region_전라', 'activity_region_제주', 'activity_region_충청', 'activity_region_

## 5. 공통 평가 함수 (고정)

MAE·RMSE·R²를 계산하는 방식이 사람마다 다르면(반올림, log 원복 여부 등) 숫자가 미묘하게 어긋납니다.
아래 함수를 그대로 가져다 쓰세요.


In [21]:
def evaluate(y_true, y_pred, label=""):
    """MAE, RMSE, R2를 계산하고 출력한 뒤 딕셔너리로 반환"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"[{label}] MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")
    return {"label": label, "MAE": mae, "RMSE": rmse, "R2": r2}

# 각 모델을 평가한 결과를 이 리스트에 계속 append해서
# 마지막에 pd.DataFrame(results)로 한 번에 비교표를 만드세요.
results = []

## 6. 스케일링 규칙 (고정)

- **트리 모델(RandomForest, LightGBM)**: 스케일링 하지 않습니다. `X_train`, `X_test` 원본을 그대로 씁니다.
- **선형회귀 / Ridge**: 아래처럼 **train으로 fit, test는 transform만** 하세요. (test로 fit하면 데이터 누수입니다)


In [22]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 선형/Ridge 모델에는 X_train_scaled / X_test_scaled를 사용하세요.
# RandomForest / LightGBM에는 X_train / X_test(스케일링 안 한 것)를 사용하세요.

print("스케일링 완료:", X_train_scaled.shape, X_test_scaled.shape)

스케일링 완료: (5811, 78) (1453, 78)


## 7. 하이퍼파라미터 탐색 범위 (고정)

완전히 똑같은 값으로 고정하면 "각자 만들어보는" 의미가 없으니, **탐색할 범위(그리드)만** 통일합니다.
이 범위 안에서만 각자 탐색하세요. 


In [23]:
RF_PARAM_GRID = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 4],
}

LGBM_PARAM_GRID = {
    "num_leaves": [15, 31, 63],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 200, 300],
}

RIDGE_PARAM_GRID = {
    "alpha": [0.1, 1.0, 10.0, 50.0],
}

---
## 체크리스트 — 모델 코드를 짜기 전에 확인하세요

- [ ] `X_train_clean.csv` / `X_test_clean.csv` / `y_train.csv` / `y_test.csv`를 그대로 썼는가 (직접 split 다시 안 했는가)
- [ ] `RANDOM_STATE = 42`를 모델·KFold 등에 전부 적용했는가
- [ ] `TARGET_A`, `TARGET_B` 이름을 정확히 그대로 썼는가
- [ ] 피처 목록(78개)을 임의로 바꾸지 않았는가
- [ ] `evaluate()` 함수를 그대로 재사용했는가
- [ ] 선형/Ridge는 스케일링된 X, 트리 모델은 원본 X를 썼는가
- [ ] 하이퍼파라미터 탐색을 정해진 그리드 범위 안에서만 했는가

전부 체크됐다면 결과를 `results` 리스트에 쌓아서 팀 전체 비교표로 합칠 준비가 된 것입니다.


In [29]:
%pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 컬럼명에서 LightGBM 충돌 특수문자(쉼표, 콜론, 따옴표, 괄호 등)를 언더바(_)로 치환
X_train.columns = [re.sub(r'[    ,:{}\[\]"]', '_', col) for col in X_train.columns]
X_test.columns = [re.sub(r'[    ,:{}\[\]"]', '_', col) for col in X_test.columns]

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV

# =========================================================
# 0-1. 결과 누적 리스트 안전 초기화
# =========================================================
if 'results' not in globals():
    results = []


# =========================================================
# 0-2. LightGBM 특수문자 충돌 방지 (컬럼명 정제)
# =========================================================
X_train.columns = [re.sub(r'[,:{}\[\]"]', '_', col) for col in X_train.columns]
X_test.columns = [re.sub(r'[,:{}\[\]"]', '_', col) for col in X_test.columns]

# =========================================================
# Step 2~4: 베이스라인 모델 학습 (출력 확인용, results에는 넣지 않음)
# =========================================================
baseline_models = {
    "LinearRegression": (LinearRegression(), X_train_scaled, X_test_scaled),
    "Ridge": (Ridge(random_state=RANDOM_STATE), X_train_scaled, X_test_scaled),
    "RandomForest": (RandomForestRegressor(random_state=RANDOM_STATE), X_train, X_test),
    "LightGBM": (LGBMRegressor(random_state=RANDOM_STATE, verbose=-1), X_train, X_test)
}

baseline_results = []

print("=== TARGET_B 베이스라인 모델 평가 진행 중 ===")
for model_name, (model, X_tr, X_te) in baseline_models.items():
    model.fit(X_tr, y_train_B)
    y_pred = model.predict(X_te)
    
    res = evaluate(y_test_B, y_pred, label=f"TARGET_B_Baseline_{model_name}")
    res["best_params"] = "Default"
    
    # [수정] baseline_results(화면 출력용)에만 저장하고 results에는 넣지 않음!
    baseline_results.append(res)

# =========================================================
# Step 5: 베이스라인 성능 비교 및 TOP 후보 모델 자동 선정
# =========================================================
df_baseline = pd.DataFrame(baseline_results)
print("\n[ TARGET_B 베이스라인 전체 비교표 ]")
print(df_baseline[["label", "MAE", "RMSE", "R2"]])

best_baseline_row = df_baseline.sort_values(by="R2", ascending=False).iloc[0]
best_model_name = best_baseline_row["label"].replace("TARGET_B_Baseline_", "")

print(f"\n 최종 선정 모델: [{best_model_name}]")

# =========================================================
# Step 6: 선정 모델 하이퍼파라미터 튜닝 (정밀 GridSearch)
# =========================================================
FINE_LGBM_PARAM_GRID = {
    "learning_rate": [0.02, 0.03, 0.04],
    "n_estimators": [200, 220, 240],
    "num_leaves": [30, 40, 50, 60]
}

print(f"\n=== [{best_model_name}] 정밀 하이퍼파라미터 튜닝 시작 ===")
lgbm_model = LGBMRegressor(random_state=RANDOM_STATE, verbose=-1)

grid_search_fine = GridSearchCV(
    estimator=lgbm_model,
    param_grid=FINE_LGBM_PARAM_GRID,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)
grid_search_fine.fit(X_train, y_train_B)

# =========================================================
# Step 7: 최종 확정 모델 예측, 후처리 및 results에만 유일하게 누적
# =========================================================

best_params = grid_search_fine.best_params_
final_model_B = grid_search_fine.best_estimator_

# 단일 최종 모델 예측 및 Cliipng 후처리
y_pred_raw = final_model_B.predict(X_test)
y_pred_final = np.clip(y_pred_raw, 0, None)

# 파라미터 식별 태그를 명시하여 최종 결과만 results에 스택
param_tag = f"lr={best_params['learning_rate']}_n={best_params['n_estimators']}_l={best_params['num_leaves']}"
final_label = f"TARGET_B_Final_{best_model_name}({param_tag})"

final_res = evaluate(y_test_B, y_pred_final, label=final_label)
final_res["best_params"] = best_params

results.append(final_res)

print(f"\n=== [최종 기록 완료] {final_label} ===")
print("최적 하이퍼파라미터:", best_params)

# =========================================================
# 튜닝 누적 기록 출력 (R2 내림차순)
# =========================================================
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
df_all_history = pd.DataFrame(results)

# ---------------------------------------------------------
# 잔차 분석 데이터 세팅
# ---------------------------------------------------------
residuals = y_test_B - y_pred_final
df_residual = pd.DataFrame({
    "y_true": y_test_B,
    "y_pred": y_pred_final,
    "residual": residuals
})

# ---------------------------------------------------------
# 피처 중요도 추출
# ---------------------------------------------------------
feature_imp = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": final_model_B.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\n[ TARGET_B 주요 피처 TOP 10 ]")
print(feature_imp.head(10))

=== TARGET_B 베이스라인 모델 평가 진행 중 ===
[TARGET_B_Baseline_LinearRegression] MAE=1.0203  RMSE=1.2824  R2=0.3268
[TARGET_B_Baseline_Ridge] MAE=1.0203  RMSE=1.2823  R2=0.3268
[TARGET_B_Baseline_RandomForest] MAE=1.0016  RMSE=1.2429  R2=0.3675
[TARGET_B_Baseline_LightGBM] MAE=0.9787  RMSE=1.2212  R2=0.3895

[ TARGET_B 베이스라인 전체 비교표 ]
                                label       MAE      RMSE        R2
0  TARGET_B_Baseline_LinearRegression  1.020340  1.282363  0.326770
1             TARGET_B_Baseline_Ridge  1.020347  1.282347  0.326787
2      TARGET_B_Baseline_RandomForest  1.001577  1.242928  0.367540
3          TARGET_B_Baseline_LightGBM  0.978664  1.221154  0.389505

 최종 선정 모델: [LightGBM]

=== [LightGBM] 정밀 하이퍼파라미터 튜닝 시작 ===
[TARGET_B_Final_LightGBM(lr=0.03_n=220_l=40)] MAE=0.9707  RMSE=1.2113  R2=0.3993

=== [최종 기록 완료] TARGET_B_Final_LightGBM(lr=0.03_n=220_l=40) ===
최적 하이퍼파라미터: {'learning_rate': 0.03, 'n_estimators': 220, 'num_leaves': 40}

[ TARGET_B 주요 피처 TOP 10 ]
                    feature

In [73]:
pd.set_option('display.max_rows', None)        
pd.set_option('display.max_columns', None)     
pd.set_option('display.max_colwidth', None)   
pd.set_option('display.width', 1000)
display(pd.DataFrame(results))

,label,MAE,RMSE,R2,best_params
0,TARGET_B_Ridge,1.020406,1.282255,0.326884,{'alpha': 10.0}
1,TARGET_B_RandomForest,0.998462,1.240027,0.370489,"{'max_depth': 20, 'min_samples_leaf': 1, 'n_estimators': 300}"
2,TARGET_B_LightGBM,0.973543,1.220423,0.390236,"{'learning_rate': 0.05, 'n_estimators': 200, 'num_leaves': 31}"
3,TARGET_B_Final_LightGBM,0.970732,1.211321,0.399298,"{'learning_rate': 0.03, 'n_estimators': 220, 'num_leaves': 40}"
4,TARGET_B_LGBM_Fine (Clipped),0.970732,1.211321,0.399298,"{'learning_rate': 0.03, 'n_estimators': 220, 'num_leaves': 40}"
5,TARGET_B_Final_LightGBM(lr=0.03_n=220_l=40),0.970732,1.211321,0.399298,"{'learning_rate': 0.03, 'n_estimators': 220, 'num_leaves': 40}"
6,TARGET_B_Final_LightGBM(lr=0.03_n=220_l=40),0.970732,1.211321,0.399298,"{'learning_rate': 0.03, 'n_estimators': 220, 'num_leaves': 40}"
7,TARGET_B_Final_LightGBM(lr=0.03_n=220_l=40),0.970732,1.211321,0.399298,"{'learning_rate': 0.03, 'n_estimators': 220, 'num_leaves': 40}"
8,TARGET_B_Final_LightGBM(lr=0.03_n=220_l=40),0.970732,1.211321,0.399298,"{'learning_rate': 0.03, 'n_estimators': 220, 'num_leaves': 40}"
9,TARGET_B_Final_LightGBM(lr=0.03_n=220_l=40),0.970732,1.211321,0.399298,"{'learning_rate': 0.03, 'n_estimators': 220, 'num_leaves': 40}"


# 가장 최적 파라미터 및 모델

In [74]:
df_results = pd.DataFrame(results)
best_row_df = df_results.sort_values(by="R2", ascending=False).head(1)

display(best_row_df)

,label,MAE,RMSE,R2,best_params
3,TARGET_B_Final_LightGBM,0.970732,1.211321,0.399298,"{'learning_rate': 0.03, 'n_estimators': 220, 'num_leaves': 40}"
